# Task 3 — Human audit (30 photo → Monet pairs, 2 raters, Cohen's κ)

Run this notebook from `task3_gan/manjot/src/`.

1. **Run cells 1–3**: they pick 30 random pairs (fixed seed 266, so both raters see identical images), copy them to
   `../human_audit/`, write `audit.html` and one blank rating sheet per rater, and show the pairs below.
2. **Each rater fills in their own sheet** (`human_audit/ratings_manjot.csv`, `human_audit/ratings_kavya.csv`) without
   discussing the scores first. In the `score_1to3` column:
   - **1 = fails**: not Monet-like, or the photo's content is badly broken
   - **2 = partial**: some painterly style, but it clearly looks like a filtered photo, or has visible artefacts
   - **3 = convincing**: reads as a Monet-style painting **and** keeps the photo's content

   Use the `note` column to say what goes wrong; those notes feed the visual failure cases in `failure_analysis.md`.
3. **Run the last cell**: it reports each rater's mean score, raw agreement, and Cohen's κ (unweighted and quadratic).

In [ ]:
import csv, glob, html, random, shutil
from pathlib import Path

SEED, N = 266, 30
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()   # task3_gan/manjot
DATA = ROOT.parent / "data"
PRED = ROOT / "outputs" / "pred_B2A"          # final submitted run-4A predictions
OUT  = ROOT / "human_audit"
RATERS = ["manjot", "kavya"]
print("predictions:", PRED, "| found", len(list(PRED.glob("*.jpg"))), "images")

In [ ]:
def find_photo(name):
    hits = [h for h in glob.glob(str(DATA / "**" / name), recursive=True) if "monet" not in h.lower()]
    return Path(hits[0]) if hits else None

preds = sorted(p.name for p in PRED.glob("*.jpg"))
random.Random(SEED).shuffle(preds)
chosen = []
for name in preds:
    src = find_photo(name)
    if src:
        chosen.append((name, src))
    if len(chosen) == N:
        break
assert len(chosen) == N, f"only matched {len(chosen)} input photos under {DATA}"

(OUT / "img").mkdir(parents=True, exist_ok=True)
rows = []
for i, (name, src) in enumerate(chosen, 1):
    sid = f"{i:02d}"
    shutil.copy(src, OUT / "img" / f"{sid}_photo.jpg")
    shutil.copy(PRED / name, OUT / "img" / f"{sid}_monet.jpg")
    rows.append((sid, name))
with open(OUT / "audit_items.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "file"]); w.writerows(rows)

for r in RATERS:                                   # never overwrites a sheet that already exists
    p = OUT / f"ratings_{r}.csv"
    if p.exists():
        print("kept existing", p.name); continue
    with open(p, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "file", "score_1to3", "note"])
        for sid, name in rows:
            w.writerow([sid, name, "", ""])

cards = "\n".join(
    f'<div style="display:inline-block;margin:6px;padding:4px;border:1px solid #ccc"><b>#{sid}</b> '
    f'<small>{html.escape(name)}</small><br><img src="img/{sid}_photo.jpg" width=256> '
    f'<img src="img/{sid}_monet.jpg" width=256></div>' for sid, name in rows)
(OUT / "audit.html").write_text(
    '<!doctype html><meta charset="utf-8"><title>Task 3 audit</title><body style="font-family:sans-serif">'
    '<h2>Left = input photo, right = generated Monet</h2>'
    '<p>1 = fails &nbsp; 2 = partial / filtered photo / artefacts &nbsp; 3 = convincing Monet with content kept</p>'
    + cards)
print(f"MADE {OUT}: {N} pairs, audit.html, ratings_manjot.csv, ratings_kavya.csv")

In [ ]:
# Show the 30 pairs here as well (left = photo, right = generated Monet); also saved as audit_grid.png
from PIL import Image
import matplotlib.pyplot as plt
fig, axes = plt.subplots(10, 6, figsize=(15, 26))
for k, (sid, name) in enumerate(rows):
    for j, kind in enumerate(["photo", "monet"]):
        ax = axes[k // 3, (k % 3) * 2 + j]
        ax.imshow(Image.open(OUT / "img" / f"{sid}_{kind}.jpg")); ax.axis("off")
        ax.set_title(f"#{sid} {kind}", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "audit_grid.png", dpi=80); plt.show()

**Stop here until both rating sheets are filled in and saved**, then run the cell below.

In [ ]:
from sklearn.metrics import cohen_kappa_score
s = {}
for r in RATERS:
    with open(OUT / f"ratings_{r}.csv") as f:
        vals = [int(x["score_1to3"]) for x in csv.DictReader(f) if x["score_1to3"].strip()]
    assert len(vals) == N and set(vals) <= {1, 2, 3}, f"{r}: need {N} scores in 1-3, got {len(vals)}"
    s[r] = vals
a, b = s[RATERS[0]], s[RATERS[1]]
lines = ["rater,mean_score,pct_3,pct_1"]
for r in RATERS:
    v = s[r]
    lines.append(f"{r},{sum(v)/N:.2f},{100*v.count(3)/N:.1f},{100*v.count(1)/N:.1f}")
lines += [f"raw_agreement,{sum(x == y for x, y in zip(a, b)) / N:.3f},,",
          f"cohen_kappa,{cohen_kappa_score(a, b):.3f},,",
          f"cohen_kappa_quadratic,{cohen_kappa_score(a, b, weights='quadratic'):.3f},,"]
(OUT / "audit_summary.csv").write_text("\n".join(lines) + "\n")
print("\n".join(lines))